In [1]:
# ============================================================
# FINMARK PREDICTIVE MODEL
# Goal: Predict whether a customer will make a purchase
# Algorithm: Logistic Regression
# ============================================================

# pandas helps us work with tables of data (like Excel but in Python)
import pandas as pd

# numpy helps us do math stuff like finding averages
import numpy as np

# these are the machine learning tools we need
from sklearn.linear_model import LogisticRegression   # the model we will use
from sklearn.model_selection import train_test_split  # splits data into training and testing
from sklearn.preprocessing import StandardScaler, LabelEncoder  # helps prepare the data
from sklearn.metrics import (
    accuracy_score,        # tells us how accurate the model is
    classification_report, # gives us a detailed performance report
    confusion_matrix,      # shows us where the model got confused
    roc_auc_score          # another way to measure how good the model is
)

# this just hides some annoying warning messages
import warnings
warnings.filterwarnings('ignore')

print('All tools loaded successfully!')

All tools loaded successfully!


In [2]:
# ============================================================
# LOAD DATASETS
# ============================================================

import os

# tell Python which folder to look in for our files
os.chdir(r'C:\Users\Matti\Documents\School')

# load each CSV file into a variable
customers_df    = pd.read_csv('customers_data.csv')
products_df     = pd.read_csv('products_data.csv')
transactions_df = pd.read_csv('transactions_data.csv')

print('Datasets loaded!')
print(f'   customers_data:    {customers_df.shape[0]} rows, {customers_df.shape[1]} columns')
print(f'   products_data:     {products_df.shape[0]} rows, {products_df.shape[1]} columns')
print(f'   transactions_data: {transactions_df.shape[0]} rows, {transactions_df.shape[1]} columns')

Datasets loaded!
   customers_data:    100 rows, 4 columns
   products_data:     20 rows, 3 columns
   transactions_data: 10000 rows, 8 columns


In [3]:
# ============================================================
# EXAMINE DATASETS — check column names, data types, missing values
# ============================================================

for name, df in [('customers_data', customers_df), ('products_data', products_df), ('transactions_data', transactions_df)]:
    print(f'\n--- {name} ---')
    print('Columns and their data types:')
    print(df.dtypes)
    print('\nHow many values are missing per column:')
    print(df.isnull().sum())


--- customers_data ---
Columns and their data types:
Company_ID        float64
Company_Name          str
Company_Profit    float64
Address               str
dtype: object

How many values are missing per column:
Company_ID        10
Company_Name       0
Company_Profit    12
Address            0
dtype: int64

--- products_data ---
Columns and their data types:
Product_ID       float64
Product_Name         str
Product_Price        str
dtype: object

How many values are missing per column:
Product_ID       2
Product_Name     0
Product_Price    0
dtype: int64

--- transactions_data ---
Columns and their data types:
Unnamed: 0          float64
Transaction_ID      float64
Company_ID          float64
Product_ID          float64
Quantity            float64
Transaction_Date        str
Product_Price       float64
Total_Cost          float64
dtype: object

How many values are missing per column:
Unnamed: 0          1000
Transaction_ID      1000
Company_ID          1000
Product_ID          1000
Q

In [4]:
# preview the first few rows of each dataset
print('--- First 5 rows of customers_data ---')
display(customers_df.head())

print('--- First 5 rows of products_data ---')
display(products_df.head())

print('--- First 5 rows of transactions_data ---')
display(transactions_df.head())

--- First 5 rows of customers_data ---


,Company_ID,Company_Name,Company_Profit,Address
0,1.0,Tech Enterprises 1,80701.0,"EDSA, Barangay 606, Pasig, Philippines"
1,2.0,Global Partners 2,80511.0,"Commonwealth Ave, Barangay 789, Taguig, Philip..."
2,3.0,Quantum Associates 3,110664.0,"Roxas Blvd, Barangay 505, Pasig, Philippines"
3,4.0,Prime Network 4,NaN,"Alabang-Zapote Rd, Barangay 202, Taguig, Phili..."
4,5.0,Elite Ventures 5,69427.0,"Ayala Avenue, Barangay 101, Makati, Philippines"


--- First 5 rows of products_data ---


,Product_ID,Product_Name,Product_Price
0,1.0,FinPredictor Suite,"?140,000"
1,2.0,MarketMinder Analytics,"?168,000"
2,3.0,TrendWise Forecaster,"?100,800"
3,4.0,CustomerScope Insights,"?123,200"
4,5.0,SalesSync Optimizer,"?84,000"


--- First 5 rows of transactions_data ---


,Unnamed: 0,Transaction_ID,Company_ID,Product_ID,Quantity,Transaction_Date,Product_Price,Total_Cost
0,0.0,1.0,88.0,6.0,NaN,2024/03/26,194379.147964,1075200.0
1,1.0,2.0,29.0,19.0,16.0,"July 09, 2024",97930.993380,1428000.0
2,2.0,NaN,28.0,18.0,6.0,04/13/2024,126095.547778,940800.0
3,3.0,4.0,85.0,12.0,12.0,09-06-2023,NaN,1008000.0
4,4.0,5.0,47.0,3.0,8.0,07/06/2021,99575.609634,705600.0


In [5]:
# ============================================================
# CLEAN customers_data
# - drop rows with no Company_ID (can't use a record with no ID)
# - fill missing Company_Profit with the median value
# - convert Company_ID from decimal (1.0) to whole number (1)
# ============================================================

customers_clean = customers_df.copy()
customers_clean = customers_clean.dropna(subset=['Company_ID'])
customers_clean['Company_Profit'] = customers_clean['Company_Profit'].fillna(customers_clean['Company_Profit'].median())
customers_clean['Company_ID'] = customers_clean['Company_ID'].astype(int)

print(f'customers_data cleaned: missing values went from {customers_df.isnull().sum().sum()} to {customers_clean.isnull().sum().sum()}')

customers_data cleaned: missing values went from 22 to 0


In [6]:
# ============================================================
# CLEAN products_data
# - drop rows with no Product_ID
# - Product_Price has a currency symbol — strip it so Python reads it as a number
# - fill any missing prices with the median price
# - convert Product_ID to whole number
# ============================================================

products_clean = products_df.copy()
products_clean = products_clean.dropna(subset=['Product_ID'])

# remove anything that isn't a number or dot from the price column
products_clean['Product_Price'] = (
    products_clean['Product_Price']
    .astype(str)
    .str.replace(r'[^\d.]', '', regex=True)
    .replace('', np.nan)
)
products_clean['Product_Price'] = pd.to_numeric(products_clean['Product_Price'], errors='coerce')
products_clean['Product_Price'] = products_clean['Product_Price'].fillna(products_clean['Product_Price'].median())
products_clean['Product_ID'] = products_clean['Product_ID'].astype(int)

print(f'products_data cleaned: missing values went from {products_df.isnull().sum().sum()} to {products_clean.isnull().sum().sum()}')

products_data cleaned: missing values went from 2 to 0


In [7]:
# ============================================================
# CLEAN transactions_data
# - remove the unnamed index column pandas added automatically
# - drop rows missing any ID column (we can't use them)
# - fill missing number columns with their median values
# - convert ID columns to whole numbers
# ============================================================

transactions_clean = transactions_df.copy()
transactions_clean = transactions_clean.drop(columns=['Unnamed: 0'], errors='ignore')
transactions_clean = transactions_clean.dropna(subset=['Transaction_ID', 'Company_ID', 'Product_ID'])

for col in ['Quantity', 'Product_Price', 'Total_Cost']:
    transactions_clean[col] = transactions_clean[col].fillna(transactions_clean[col].median())

for col in ['Transaction_ID', 'Company_ID', 'Product_ID']:
    transactions_clean[col] = transactions_clean[col].astype(int)

print(f'transactions_data cleaned: missing values went from {transactions_df.isnull().sum().sum()} to {transactions_clean.isnull().sum().sum()}')

transactions_data cleaned: missing values went from 7000 to 0


In [8]:
# ============================================================
# MERGE ALL 3 DATASETS INTO ONE
# - join transactions with customers using Company_ID
# - then join with products using Product_ID
# ============================================================

df = transactions_clean.merge(customers_clean, on='Company_ID', how='left')
df = df.merge(products_clean[['Product_ID', 'Product_Name']], on='Product_ID', how='left')

print(f'Combined dataset shape: {df.shape[0]} rows, {df.shape[1]} columns')
display(df.head())

Combined dataset shape: 7277 rows, 11 columns


,Transaction_ID,Company_ID,Product_ID,Quantity,Transaction_Date,Product_Price,Total_Cost,Company_Name,Company_Profit,Address,Product_Name
0,1,88,6,10.0,2024/03/26,194379.147964,1075200.0,Elite Consulting 88,75950.0,"EDSA, Barangay 456, Taguig, Philippines",RevenueVue Dashboard
1,2,29,19,16.0,"July 09, 2024",97930.993380,1428000.0,Sky Industries 29,61952.0,"Edsa, brgy. 606, makati, philippines!",EcoNomix Modeler
2,4,85,12,12.0,09-06-2023,130556.442432,1008000.0,Green Ventures 85,113470.0,"EDSA, Barangay 707, Cebu City, Philippines",BudgetMaster Pro
3,5,47,3,8.0,07/06/2021,99575.609634,705600.0,Green Industries 47,31130.0,"Taft Ave, Barangay 707, Mandaluyong, Philippines",TrendWise Forecaster
4,6,80,11,4.0,2021/07/12,160658.675350,627200.0,Green Partners 80,111227.0,"Commonwealth Ave, Barangay 202, Manila, Philip...",OptiFlow Automation


In [9]:
# ============================================================
# FIX LEFTOVER MISSING VALUES AFTER MERGE
# Some rows couldn't find a match when joining — those came back as NaN
# We fill those here so the model doesn't crash
# ============================================================

# fill missing numbers with the column median
for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())

# fill missing text with 'Unknown'
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna('Unknown')

print(f'Leftover missing values after merge: {df.isnull().sum().sum()}')
print('All clean and ready to go!')

Leftover missing values after merge: 0
All clean and ready to go!


In [10]:
# ============================================================
# CREATE TARGET VARIABLE (the answer column)
# purchased = 1 if Quantity > 0 (a real purchase happened)
# purchased = 0 if Quantity = 0 (no purchase)
# ============================================================

df['purchased'] = (df['Quantity'] > 0).astype(int)

print('How many purchased (1) vs not purchased (0):')
print(df['purchased'].value_counts())
print(f'\nOverall purchase rate: {df["purchased"].mean()*100:.1f}%')

How many purchased (1) vs not purchased (0):
purchased
1    7172
0     105
Name: count, dtype: int64

Overall purchase rate: 98.6%


In [11]:
# ============================================================
# SELECT FEATURES (input columns for the model)
# These are the columns that help the model learn purchasing behaviour
# ============================================================

feature_cols = [
    'Company_Profit',  # how profitable the company is
    'Product_Price',   # how expensive the product is
    'Quantity',        # how many items were in the transaction
    'Total_Cost',      # the total amount spent
    'Product_Name',    # which product it was
]

# only keep columns that actually exist in our dataset
feature_cols = [col for col in feature_cols if col in df.columns]

print(f'Using these {len(feature_cols)} features to train the model:')
for col in feature_cols:
    print(f'   - {col}')

# X = input data (what the model learns from)
# y = answer column (what the model is trying to predict)
X = df[feature_cols].copy()
y = df['purchased']

Using these 5 features to train the model:
   - Company_Profit
   - Product_Price
   - Quantity
   - Total_Cost
   - Product_Name


In [12]:
# ============================================================
# ENCODE TEXT COLUMNS INTO NUMBERS
# Machine learning models only work with numbers, not text
# LabelEncoder assigns a number to each unique text value
# ============================================================

le = LabelEncoder()
text_columns = X.select_dtypes(include='object').columns.tolist()

for col in text_columns:
    X[col] = le.fit_transform(X[col].astype(str))

print(f'Converted text columns to numbers: {text_columns}')

# final check before training
print(f'\nFinal check — missing values in input data: {X.isnull().sum().sum()}')
print('Ready to train!' if X.isnull().sum().sum() == 0 else 'Warning: still have missing values!')

Converted text columns to numbers: ['Product_Name']

Final check — missing values in input data: 0
Ready to train!


In [13]:
# ============================================================
# SPLIT DATA INTO TRAINING AND TESTING SETS
# 80% of data is used to train the model
# 20% is kept aside to test how well it performs
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set:  {X_train.shape[0]} rows (model learns from these)')
print(f'Test set:      {X_test.shape[0]} rows (we test the model on these)')

Training set:  5821 rows (model learns from these)
Test set:      1456 rows (we test the model on these)


In [14]:
# ============================================================
# SCALE THE FEATURES
# Some columns have very large numbers, some have small numbers
# Scaling puts them all on the same range so the model isn't confused
# ============================================================

scaler = StandardScaler()

# fit and transform the training data
X_train_scaled = scaler.fit_transform(X_train)

# only transform the test data (never fit on test data)
X_test_scaled = scaler.transform(X_test)

print('Features scaled and ready!')

Features scaled and ready!


In [15]:
# ============================================================
# TRAIN THE LOGISTIC REGRESSION MODEL
# The model looks at the training data and learns patterns
# that help it predict whether a purchase will happen or not
# ============================================================

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

print('Model trained successfully!')

Model trained successfully!


In [16]:
# ============================================================
# GENERATE PREDICTIONS AND EVALUATE THE MODEL
# We ask the model to predict on the test set
# then compare its predictions to the real answers
# ============================================================

# get predictions
y_pred      = model.predict(X_test_scaled)
y_pred_prob = model.predict_proba(X_test_scaled)[:, 1]  # confidence score per prediction

# calculate scores
accuracy = accuracy_score(y_test, y_pred)
roc_auc  = roc_auc_score(y_test, y_pred_prob)

# print results
print('=' * 45)
print('         MODEL EVALUATION RESULTS')
print('=' * 45)
print(f'  Accuracy:      {accuracy*100:.2f}%')
print(f'  ROC-AUC Score: {roc_auc:.4f}  (closer to 1.0 is better)')
print('=' * 45)
print('\nDetailed Report:')
print(classification_report(y_test, y_pred, target_names=['Not Purchased', 'Purchased']))
print('Confusion Matrix (rows = actual, columns = predicted):')
print(confusion_matrix(y_test, y_pred))

         MODEL EVALUATION RESULTS
  Accuracy:      99.59%
  ROC-AUC Score: 1.0000  (closer to 1.0 is better)

Detailed Report:
               precision    recall  f1-score   support

Not Purchased       1.00      0.71      0.83        21
    Purchased       1.00      1.00      1.00      1435

     accuracy                           1.00      1456
    macro avg       1.00      0.86      0.92      1456
 weighted avg       1.00      1.00      1.00      1456

Confusion Matrix (rows = actual, columns = predicted):
[[  15    6]
 [   0 1435]]


In [17]:
# ============================================================
# FEATURE IMPORTANCE
# Each feature gets a score — higher means it had more influence
# on predicting whether a purchase happened or not
# ============================================================

importance_table = pd.DataFrame({
    'Feature': feature_cols,
    'Score': model.coef_[0]
}).sort_values('Score', ascending=False)

print('Feature importance scores (higher = more influence on predicting a purchase):')
print(importance_table.to_string(index=False))

Feature importance scores (higher = more influence on predicting a purchase):
       Feature    Score
      Quantity 9.304263
    Total_Cost 0.683281
Company_Profit 0.195926
 Product_Price 0.085573
  Product_Name 0.074707


In [18]:
# ============================================================
# SAVE ALL FILES
# ============================================================

# save the model predictions
results_df = X_test.copy()
results_df['actual']               = y_test.values   # what actually happened
results_df['predicted']            = y_pred           # what the model predicted
results_df['purchase_probability'] = y_pred_prob      # how confident the model was
results_df.to_csv('finmark_predictions.csv', index=False)

# save the cleaned datasets
customers_clean.to_csv('customers_data_clean.csv', index=False)
products_clean.to_csv('products_data_clean.csv', index=False)
transactions_clean.to_csv('transactions_data_clean.csv', index=False)

print('All files saved to C:\\Users\\Matti\\Documents\\School')
print('')
print('Files saved:')
print('   finmark_predictions.csv     <- model results')
print('   customers_data_clean.csv    <- cleaned customers')
print('   products_data_clean.csv     <- cleaned products')
print('   transactions_data_clean.csv <- cleaned transactions')

All files saved to C:\Users\Matti\Documents\School

Files saved:
   finmark_predictions.csv     <- model results
   customers_data_clean.csv    <- cleaned customers
   products_data_clean.csv     <- cleaned products
   transactions_data_clean.csv <- cleaned transactions
